## Data explore and description

### By:
Gabriel Múnera González

### Date:
2026-08-18

### Description:

Data overview and exploration to check data types and fix any issue with the data types.

this is in other to do a correct data analysis and visualization of the data.


## 📚 Import  libraries and define useful functions

In [ ]:
# base libraries for data science
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa

# -----------Variables to put in config module----------------
regex = {
    "carat": r"^[1-9](\.[0-9]+)?$",  # numeric with decimals
    "cut": r"^[A-Za-z]+\s*[A-Za-z]*$",  # categorical alphabetic
    "color": r"^[A-Ja-j]$",  # single letter (A-J)
    "clarity": r"^[A-Za-z]{1,3}[0-9]{0,1}$",  # combinations like SI1, VS2, etc.
    "depth": r"^[0-9]+\.[0-9]+$",  # numeric with decimals
    "table": r"^[0-9]+\.[0-9]+$",  # numeric with decimals
    "price": r"^[1-9][0-9]*$",  # positive integer
    "x": r"^(?=[0-9]{0,3}\.[0-9]+$)(?!0*\.0*$)[0-9]{0,3}\.[0-9]+$",  # positive numeric with decimals
    "y": r"^(?=[0-9]{0,3}\.[0-9]+$)(?!0*\.0*$)[0-9]{0,3}\.[0-9]+$",  # positive numeric with decimals
    "z": r"^(?=[0-9]{0,3}\.[0-9]+$)(?!0*\.0*$)[0-9]{0,3}\.[0-9]+$",  # positive numeric with decimals
}

validator = {
    "carat": 70,
    "cut": 153,
    "color": 227,
    "clarity": 277,
    "depth": 307,
    "table": 251,
    "price": 273,
    "x": 142,
    "y": 147,
    "z": 54,
}

dtypes = {
    "carat": np.float32(),
    "cut": "category",
    "color": "category",
    "clarity": "category",
    "depth": np.float32(),
    "table": np.float32(),
    "price": np.int64(),
    "x": np.float32(),
    "y": np.float32(),
    "z": np.float32(),
}


def get_all_masks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace all characters in the DataFrame with their corresponding mask characters.
    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame to be masked.
    Returns
    -------
    pd.DataFrame
        A DataFrame with all characters replaced by their corresponding mask characters.
    """
    df_string = df.astype({col: str for col in df.columns})
    regexs = {"[A-Z]": "L", "[a-z]": "l", "[0-9]": "D", r"\s": "s"}
    for regex, replacement in regexs.items():
        df_string = pd.concat(
            [
                df_string[col].str.replace(regex, replacement, regex=True)
                for col in df_string.columns
            ],
            axis=1,
        )
    return df_string


# --- Cleaning atypical values by column ---
def clean_column_values(df: pd.DataFrame, regex: dict) -> pd.DataFrame:
    """
    Clean the DataFrame by replacing atypical values with NaN based on regex patterns.
    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame to be cleaned.
    regex : dict
        A dictionary where keys are column names and values are regex patterns for valid values.
    Returns
    -------
    pd.DataFrame
        The cleaned DataFrame with atypical values replaced by NaN.
    """
    for column in df.columns:
        if column in regex:
            pattern = regex[column]
            df.loc[~df[column].astype(str).str.match(pattern), column] = pd.NA
    return df


def get_report_masks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate a report of the masked DataFrame, showing the value counts for each column.
    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame to generate the report from.
    Returns
    -------
    str
        A markdown table containing the value counts for each column in the masked DataFrame.
    """
    masked_df = get_all_masks(df)
    report = {}
    for col in masked_df.columns:
        report[col] = masked_df[col].value_counts(dropna=False)
    report = pd.DataFrame(report)
    index = report.index.to_series()
    index.loc[index.isna()] = "NaN"
    report.index = index
    report = report.fillna(0).astype(int)
    return report


def classify_pattern(pattern: str) -> str:
    """Classify pattern into numeric, alphabetic, alphanumeric."""
    if pattern is None or pattern == "NaN":
        return "Missing"
    if all(ch in ["D", ".", "-"] for ch in pattern):
        return "Numeric"
    if all(ch == "L" or ch.islower() for ch in pattern):
        return "Alphabetic"
    if "L" in pattern and "D" in pattern:
        return "Alphanumeric"
    return "Unknown"


def get_report_cleaning(mask_report: pd.DataFrame, atipic_threshold: float = 0.05) -> str:
    """
    Consolidate the cleaning report from the mask report. This function creates a markdown that shows
    the get_report_masks presenting the typical value (example: numerical with two decimals) and the
    atypical value (example: numerical with three decimals) and the percentage of atypical values
    for each column.
    Parameters
    ----------
    mask_report : pd.DataFrame
        The markdown report generated in get_report_masks.
    atipic_threshold : float, optional
    Returns
    -------
    str
        A markdown report to place in /notebooks/8-reports/1-exploration.
    """
    markdown = ["# Cleaning Report\n"]
    # Add the original mask table
    markdown.append(mask_report.to_markdown())

    # Iterate over columns (skip index)
    for col in mask_report.columns:
        col_counts = mask_report[col]
        total = col_counts.sum()
        prevalent_pattern = str(col_counts.idxmax())
        prevalent_count = col_counts.max()

        # Classification
        classification = classify_pattern(prevalent_pattern)
        if classification == "Numeric":
            classification = (
                "Continuous numeric" if "." in prevalent_pattern else "Discrete numeric"
            )
        elif classification == "Alphabetic":
            classification = "Alphabetic categorical"
        elif classification == "Alphanumeric":
            classification = "Alphanumeric categorical"

        # Atypical values
        atypical = col_counts[
            (col_counts / mask_report.sum().iloc[0] < atipic_threshold) & (col_counts > 0)
        ]
        atypical_lines = []
        for pat, cnt in atypical.items():
            if pat == "NaN":
                continue
            atypical_lines.append(f"  - `{pat}` ({cnt} occurrence{'s' if cnt > 1 else ''})")

        # NA values
        na_count = col_counts.get("NaN", 0)
        na_pct = (na_count / total * 100) if total > 0 else 0

        # Build section
        section = f"""
---
## Column: {col}
- **Prevalent pattern:** `{prevalent_pattern}` ({prevalent_count} occurrences).
- **Classification:** {classification}.
- **Atypical values:**
{chr(10).join(atypical_lines) if atypical_lines else "  - None"}
- **NA values:** {na_count} (≈{na_pct:.2f}%).
- **Conclusion:** Predominantly {classification.lower()}, with {"rare anomalies" if len(atypical_lines) > 0 else "no anomalies"} and a {"small" if na_pct < 1 else "moderate"} proportion of missing values.
"""
        markdown.append(section)

    return "\n".join(markdown)

## 💾 Load data

In [2]:
# data directory path
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

diamantes_df = pd.read_csv(DATA_DIR / "01_raw/diamantes.csv")

## 📊 Data description

In [3]:
diamantes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55126 entries, 0 to 55125
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    55059 non-null  object 
 1   cut      54977 non-null  object 
 2   color    54903 non-null  object 
 3   clarity  54852 non-null  object 
 4   depth    54821 non-null  object 
 5   table    54880 non-null  object 
 6   price    54855 non-null  object 
 7   x        54986 non-null  float64
 8   y        54981 non-null  float64
 9   z        55074 non-null  float64
dtypes: float64(3), object(7)
memory usage: 4.2+ MB


In [4]:
diamantes_df.sample(10)

,carat,cut,color,clarity,depth,table,price,x,y,z
11838,1.2,Premium,I,SI1,61.4,57.0,5098,6.87,6.80,4.20
18646,1.21,Premium,G,VS2,62.2,60.0,7611,6.79,6.75,4.21
51717,0.7,Good,F,SI1,57.2,63.0,2401,5.84,5.87,3.35
5829,0.92,Premium,G,SI2,59.4,60.0,3916,6.34,6.31,3.76
14255,1.0,Ideal,D,SI1,60.6,56.0,5775,6.50,6.54,3.95
39698,0.4,Ideal,H,VVS1,62.3,54.0,1088,4.77,4.74,2.96
17768,1.59,Ideal,J,SI2,62.1,54.0,7155,7.51,7.44,4.64
42172,0.5,Ideal,F,SI1,62.7,56.0,1286,5.12,5.05,3.19
18600,1.12,Very Good,E,VS2,62.8,57.0,7589,6.61,6.64,4.16
53032,0.7,Premium,H,VVS1,58.3,60.0,2603,5.83,5.80,3.39


## Mask analysis to identify columns with incorrect data types and/or outliers

We found that, even though the columns are numerical, they were read as string values. This is a common issue when loading data, especially if there are missing values or non-numeric characters in the columns.

There is possible too that the data contains outliers that need to be addressed.

To catch this issues (where it is the most evident), we'll use the function `get_all_masks` to identify which columns are affected and then convert them to the appropriate numeric type.

In [5]:
masked_diamantes_df = get_report_masks(diamantes_df)
print(get_report_cleaning(masked_diamantes_df))

# Cleaning Report

|              |   carat |   cut |   color |   clarity |   depth |   table |   price |     x |     y |     z |
|:-------------|--------:|------:|--------:|----------:|--------:|--------:|--------:|------:|------:|------:|
| -DDDDD.D     |       0 |     0 |       0 |         0 |       0 |       0 |       0 |     0 |     0 |     1 |
| -DDDDDD.D    |       0 |     0 |       0 |         0 |       0 |       0 |       0 |     0 |     1 |     0 |
| -DDDDDDDDD.D |       0 |     0 |       0 |         0 |       0 |       0 |       0 |     1 |     0 |     0 |
| D.D          |   13673 |     0 |       0 |         0 |       0 |       0 |       0 |  5398 |  5662 |  5556 |
| D.DD         |   41383 |     0 |       0 |         0 |       0 |       0 |       0 | 49580 | 49312 | 49515 |
| DD           |       0 |     0 |       2 |         0 |       0 |       0 |       0 |     0 |     0 |     0 |
| DD.D         |       0 |     0 |       0 |         0 |   54819 |   54875 |       0 |     1 

## Data coleanup and validation

In [8]:
diamantes_df = clean_column_values(diamantes_df, regex)
# --- Validation ---
print("Count of NaN values per column:")
print(diamantes_df.isna().sum())
print("\ncompliance with the estimated nan quantity")
pd.DataFrame(validator, index=["validator"]).T.rename(
    columns={"validator": "count"}
) == diamantes_df.isna().sum().to_frame().rename(columns={0: "count"})

Count of NaN values per column:
carat      35854
cut          153
color        227
clarity      277
depth        307
table        251
price        273
x            150
y            154
z             74
dtype: int64

compliance with the estimated nan quantity


,count
carat,False
cut,True
color,True
clarity,True
depth,True
table,True
price,True
x,False
y,False
z,False


## Null values

In this dataset the null values are typed `np.na`. We'll drop the rows with null values to ensure that our analysis is based on complete data. This is important because null values can skew the results of our analysis and lead to incorrect conclusions. We also take this decision due to the fact that the dataset is large enough to handle the loss of some rows without significantly impacting the overall analysis.

In [11]:
diamantes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55126 entries, 0 to 55125
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    19272 non-null  object 
 1   cut      54973 non-null  object 
 2   color    54899 non-null  object 
 3   clarity  54849 non-null  object 
 4   depth    54819 non-null  object 
 5   table    54875 non-null  object 
 6   price    54853 non-null  object 
 7   x        54976 non-null  float64
 8   y        54972 non-null  float64
 9   z        55052 non-null  float64
dtypes: float64(3), object(7)
memory usage: 4.2+ MB


In [14]:
diamantes_df = diamantes_df.dropna().reset_index(drop=True)
diamantes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19231 entries, 0 to 19230
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    19231 non-null  object 
 1   cut      19231 non-null  object 
 2   color    19231 non-null  object 
 3   clarity  19231 non-null  object 
 4   depth    19231 non-null  object 
 5   table    19231 non-null  object 
 6   price    19231 non-null  object 
 7   x        19231 non-null  float64
 8   y        19231 non-null  float64
 9   z        19231 non-null  float64
dtypes: float64(3), object(7)
memory usage: 1.5+ MB


## Remove columns

All the collumns are relevant for the analysis, so no columns will be removed.

### Categorical variables
#### Ordinal
- `clarity`: a measurement of how clear the diamond
    - 1I1 (worst)
    - SI2
    - SI1
    - VS2
    - VS1
    - VVS2
    - VVS1
    - IF (best)
- `color`: a measurement of how clear the diamond
    - D (best)
    - E
    - F
    - G
    - H
    - I
    - J (worst)
- `cut`: a measurement of how well the diamond has been cut
    - Ideal (best)
    - Premium
    - Very Good
    - Good
    - Fair (worst)

#### Nominal

No nominal categorical variables were found in the dataset.

### Numerical variables
#### Discrete
No discrete numerical variables were found in the dataset.

#### Continuous

- `carat`: weight of the diamond (0.2--5.01)
- `depth`: total depth percentage = z / mean(x, y) = 2 * z / (x + y) (43--79)
- `table`: width of top of diamond relative to widest point (43--95)
- `price`: price in US dollars (\$326--\$18,823)
- `x`: length in mm (0--10.74)
- `y`: width in mm (0--58.9)
- `z`: depth in mm (0--31.8)

### Boolean variables
No boolean variables were found in the dataset.

## String variables

No string variables were found in the dataset.

## Convert data types

In [15]:
diamantes_df = diamantes_df.astype(dtypes)
diamantes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19231 entries, 0 to 19230
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    19231 non-null  float32 
 1   cut      19231 non-null  category
 2   color    19231 non-null  category
 3   clarity  19231 non-null  category
 4   depth    19231 non-null  float32 
 5   table    19231 non-null  float32 
 6   price    19231 non-null  int64   
 7   x        19231 non-null  float32 
 8   y        19231 non-null  float32 
 9   z        19231 non-null  float32 
dtypes: category(3), float32(6), int64(1)
memory usage: 658.3 KB


In [16]:
schema = pa.Table.from_pandas(diamantes_df).schema

## Finding atipical values in physical properties (`x`, `y`, `z`, `carat`, `depth`, `table`, `price`) 

In [20]:
diamantes_df.describe()

,carat,depth,table,price,x,y,z
count,19231.000000,19231.000000,19231.000000,19231.000000,19231.000000,19231.000000,19231.000000
mean,1.321377,61.782955,57.918308,8101.184806,6.979585,6.973261,4.308378
std,0.366815,1.594663,2.198251,3917.475077,0.604243,0.707423,0.375342
min,1.000000,43.000000,43.000000,1262.000000,5.720000,4.110000,1.070000
25%,1.030000,61.000000,56.000000,5006.000000,6.500000,6.500000,4.020000
50%,1.200000,61.900002,58.000000,6794.000000,6.790000,6.780000,4.190000
75%,1.510000,62.599998,59.000000,10417.500000,7.360000,7.350000,4.550000
max,5.010000,78.199997,95.000000,18823.000000,10.740000,58.900002,8.060000


In [ ]:
columns = ["carat", "depth", "table", "price", "x", "y", "z"]

for col in columns:
    num_weird_vallues = len(diamantes_df[diamantes_df[col] <= 0])
    print(f"column: {col} has {num_weird_vallues} weird values (<=0)")

column: carat has 0 weird values (<=0)
column: depth has 0 weird values (<=0)
column: table has 0 weird values (<=0)
column: price has 0 weird values (<=0)
column: x has 0 weird values (<=0)
column: y has 0 weird values (<=0)
column: z has 0 weird values (<=0)


###  💾 Save dataframe with data types

In [22]:
diamantes_df.to_parquet(
    DATA_DIR / "02_intermediate/diamantes_type_fixed.parquet",
    index=False,
    schema=schema,
)

## 📊 Analysis of Results

All the columns have been converted to the appropriate data types, and any issues with outliers have been addressed. The dataset is now ready for further analysis.


## 📖 References

- <https://pandas.pydata.org/docs/user_guide/pyarrow.html>